<a href="https://colab.research.google.com/github/mariliazago0/fundamentos01/blob/main/notebooks/encontro01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!pip install tinydb

In [37]:
# programas/bibliotecas utilizados no script/codigo
import httpx # Responsável pelas requisições web
from bs4 import BeautifulSoup # Responsável por realizar o web scraping (coletar os dados)
from tinydb import TinyDB, Query

In [38]:
def inserir_no_banco(dados, link_noticia):
  arquivo_banco_dados = "nota_china.json"
  db = TinyDB(arquivo_banco_dados)


  # Evitar dados repetidos no banco
  Buscar = Query()
  verificar_link = db.contains(Buscar.link == link_noticia)

  if not verificar_link:
    print("Inserindo nova informação no banco")
    db.insert(dados)
  else:
    print("Link já existe no banco. Esta informação não será inserida novamente")

In [39]:
# Variável e tipos de dados (string, lista, numero)
paginas = ["https://english.news.cn/list/china-science.htm"]

def acessa_pagina (link):
  print (f"Estamos na pagina:{link}")

  # Define headers para a requisição, simulando um navegador
  headers = {
      'User-Agent': "Mozilla/50 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
      'Accept-Language': 'en-US,en;q=0.9',
      'Accept-Encoding': 'gzip, deflate, br',
      'Connection': 'keep-alive',
  }

  timeout = httpx.Timeout(connect=20.0, read=30.0, write=20.0, pool=10.0)
  pag_web = httpx.get(link, headers=headers, timeout=timeout)
  bs = BeautifulSoup(pag_web, "html.parser")
  return bs

# loop for
# beautifulsoap >> find e find_all

for pagina in paginas:
  pagina_inteira = acessa_pagina(pagina)
  lista_noticias = pagina_inteira.find("div", attrs={"class": "part"}).find_all("div", attrs={"class": "item"})
  for noticia in lista_noticias:

    # titulo
    try:
      titulo = noticia.find("div", attrs={"class": "tit"}).text.strip()
      print(titulo)
    except:
        titulo = ""

    #link
    try:
      link_noticia = noticia.a["href"]
      print(link_noticia)
    except:
      link_noticia = ""
    print("###")

    # data
    # horário
    data = ""
    hora = ""
    data_hora_elements = noticia.find_all("span",attrs={"class": "time"})
    if data_hora_elements:
      data_hora_text = data_hora_elements[0].text.strip()
      partes_tempo = data_hora_text.split(" ")
      data = partes_tempo[0] if len(partes_tempo) > 0 else ""
      hora = partes_tempo[1] if len(partes_tempo) > 1 else ""
    print(data)
    print(hora)

    conteudo = acessa_pagina (link_noticia)
    paragrafos = conteudo.find("div", attrs={"id" : "detailContent"}).find_all("p")
    lista_paragrafos = []
    for paragrafo in paragrafos:
      lista_paragrafos.append(paragrafo.text.strip())
    print(lista_paragrafos)
    # função para inserir dados coletados no banco
    dados = {
        "titulo": titulo,
        "link": link_noticia,
        "data": data,
        "hora": hora,
        "paragrafo": lista_paragrafos
    }
    inserir_no_banco(dados,link_noticia)

Estamos na pagina:https://english.news.cn/list/china-science.htm
China Focus: Chinese scientists develop "Jiuzhang 4.0," setting new world record in quantum computing2026-05-14 00:10:45
../20260514/365a07ee40354bf3a7dcffe671b1afba/c.html
###
2026-05-14
00:10:45
Estamos na pagina:../20260514/365a07ee40354bf3a7dcffe671b1afba/c.html


UnsupportedProtocol: Request URL is missing an 'http://' or 'https://' protocol.